# EviChange — fine-tune the Geo-VLM (Phase 4b, D17)

Trains two small **LoRA adapters** on Qwen2.5-VL-7B-Instruct (the base weights stay frozen):
- `check`: before/after chips → change / no change + change types (the D15 region check);
- `gated`: chips + the box evidence → the gated report JSON (`agents/gated.py`).

Training data: the hand-labelled development boxes (never the 30 labelled sites, nothing within 500 m of
them); the 80 D15 boxes are held out for evaluation.

**Before you start** (on your computer): `python scripts\finetune_build.py`, then upload everything in
`data\finetune\` (`chips.zip` and the four `.jsonl` files) to Google Drive, folder `evichange_ft_data`.

**Runtime:** `Runtime → Change runtime type` → **A100 GPU**. Expect about 1–2 h for both adapters.
Adapters and predictions are saved to Drive (`evichange_ft/`), so a disconnect loses at most the current step.

## Step 1 — Install, mount Drive, get the code and the data

In [ ]:
!pip -q install -U "transformers>=4.57" peft accelerate pillow
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/EviChange && git clone -q --depth 1 https://github.com/nachikethpt/EviChange /content/EviChange
!mkdir -p /content/ft && cp /content/drive/MyDrive/evichange_ft_data/*.jsonl /content/ft/
!cd /content/ft && unzip -q -o /content/drive/MyDrive/evichange_ft_data/chips.zip
!wc -l /content/ft/*.jsonl
import torch; print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), 'GB')

## Step 2 — Baseline: the untrained model on the held-out boxes (the number to beat)

In [ ]:
%cd /content/EviChange
!python scripts/finetune_train.py eval --task check --data /content/ft --base-only

## Step 3 — Train and evaluate the region-check adapter

In [ ]:
!python scripts/finetune_train.py train --task check --data /content/ft --out /content/drive/MyDrive/evichange_ft
!python scripts/finetune_train.py eval  --task check --data /content/ft --adapter /content/drive/MyDrive/evichange_ft/check

## Step 4 — Train and evaluate the gated-report adapter

In [ ]:
!python scripts/finetune_train.py eval  --task gated --data /content/ft --base-only
!python scripts/finetune_train.py train --task gated --data /content/ft --out /content/drive/MyDrive/evichange_ft
!python scripts/finetune_train.py eval  --task gated --data /content/ft --adapter /content/drive/MyDrive/evichange_ft/gated

## After training

Copy `predictions_*.jsonl` and the adapter folders from Drive `evichange_ft/` back to the repo
(`data/finetune/`); `scripts/finetune_build.py` scoring and the D17 tables are made on your computer.
The server notebook can load an adapter (`ADAPTER` in its Step 2) to run the fine-tuned model in the app
and for the 30 sites after the freeze.

## Stop — release the GPU

In [ ]:
from google.colab import runtime
runtime.unassign()